# ========================================
# Интеллектуальный помощник по электронной почте (EmailSmartAssistant)# ========================================

In [1]:
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import json
import re
import os
from datetime import datetime, timedelta
from collections import Counter
import jieba
from langdetect import detect
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

ModuleNotFoundError: No module named 'jieba'

# ========================================
# 0. Настройте параметры LLM# ========================================

In [ ]:
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "your_api_key_here"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

# ========================================
# 1. Определите инструменты обработки электронной почты# ========================================

In [ ]:
class EmailClassificationTool(Tool):
    """Инструмент интеллектуальной классификации почты"""
    
    def __init__(self):
        super().__init__(
            name="email_classification",
            description="Автоматическая классификация типа письма, приоритета и типа отправителя по содержимому"
        )
        
        # Загрузка правил классификации
        try:
            with open('config/email_config.json', 'r', encoding='utf-8') as f:
                config = json.load(f)
                self.classification_rules = config.get('classification_rules', {})
                self.priority_rules = config.get('priority_rules', {})
        except FileNotFoundError:
            # Правила классификации по умолчанию
            self.classification_rules = {
                'work_keywords': ['会议', '项目', '工作', '任务', '汇报', 'meeting', 'project', 'work', 'task', 'urgent'],
                'customer_keywords': ['客户', '咨询', '购买', '服务', 'customer', 'inquiry', 'purchase', 'service'],
                'personal_keywords': ['个人', '家庭', '朋友', 'personal', 'family', 'friend', '聚餐'],
                'spam_keywords': ['广告', '推广', '营销', '优惠', 'advertisement', 'promotion', 'marketing', '折扣']
            }
            self.priority_rules = {
                'high_priority_keywords': ['紧急', 'urgent', 'asap', '重要', 'important'],
                'low_priority_keywords': ['通知', 'newsletter', 'notification', '订阅']
            }
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Классификация письма и возврат результата"""
        subject = parameters.get("subject", "")
        body = parameters.get("body", "")
        sender = parameters.get("sender", "")
        
        if not subject and not body:
            return "Ошибка: тема и содержимое письма не могут быть пустыми одновременно"
        
        # Объединить текстовое содержимое для анализа        text_content = f"{subject} {body}".lower()
        
        # Проверить спам
        spam_score = sum(1 for keyword in self.classification_rules['spam_keywords'] 
                        if keyword in text_content)
        if spam_score >= 2:
            classification = {'type': 'spam', 'priority': 'low', 'sender_type': 'external'}
        else:
            # Подсчитайте баллы для каждого типа
            work_score = sum(1 for keyword in self.classification_rules['work_keywords'] 
                            if keyword in text_content)
            customer_score = sum(1 for keyword in self.classification_rules['customer_keywords'] 
                               if keyword in text_content)
            personal_score = sum(1 for keyword in self.classification_rules['personal_keywords'] 
                               if keyword in text_content)
            
            # Определение типа письма
            scores = {'work': work_score, 'customer': customer_score, 'personal': personal_score}
            email_type = max(scores, key=scores.get) if max(scores.values()) > 0 else 'other'
            
            # Расставить приоритеты
            priority = 'medium'  # Средний приоритет по умолчанию
            if any(word in text_content for word in self.priority_rules['high_priority_keywords']):
                priority = 'high'
            elif any(word in text_content for word in self.priority_rules['low_priority_keywords']):
                priority = 'low'
            
            # Определить тип отправителя
            sender_lower = sender.lower()
            if 'company.com' in sender_lower or 'corp.com' in sender_lower:
                sender_type = 'colleague'
            elif 'noreply' in sender_lower or 'no-reply' in sender_lower:
                sender_type = 'system'
            elif email_type == 'customer':
                sender_type = 'customer'
            else:
                sender_type = 'external'
            
            classification = {
                'type': email_type,
                'priority': priority,
                'sender_type': sender_type
            }
        
        return json.dumps(classification, ensure_ascii=False, indent=2)
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="subject",
                type="string",
                description="Тема письма",
                required=False
            ),
            ToolParameter(
                name="body",
                type="string",
                description="Текст письма",
                required=False
            ),
            ToolParameter(
                name="sender",
                type="string",
                description="Адрес электронной почты отправителя",
                required=True
            )
        ]

In [ ]:
class InfoExtractionTool(Tool):
    """Инструмент извлечения ключевой информации"""
    
    def __init__(self):
        super().__init__(
            name="info_extraction",
            description="Извлечение дат, времени, контактов, задач и другой ключевой информации из письма"
        )
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Извлечение ключевой информации"""
        body = parameters.get("body", "")
        
        if not body:
            return "Ошибка: содержимое письма не может быть пустым"
        
        # Дата извлечения
        date_patterns = [
            r'\d{4}-\d{1,2}-\d{1,2}',  # 2024-01-15
            r'\d{1,2}мес.\d{1,2}дн.',      #15 января            r'\d{1,2}/\d{1,2}',        # 1/15
            r'\d{1,2}-\d{1,2}'         # 1-15
        ]
        
        dates = []
        for pattern in date_patterns:
            dates.extend(re.findall(pattern, body))
        
        # Время экстракции
        time_patterns = [
            r'\d{1,2}:\d{2}',          # 14:30
            r'\d{1,2}ч.\d{0,2}мин.?',     #2:30            r'\d{1,2}\s*PM',           # 2 PM
            r'\d{1,2}\s*AM'            # 9 AM
        ]
        
        times = []
        for pattern in time_patterns:
            times.extend(re.findall(pattern, body))
        
        # Извлечь контактную информацию
        phones = re.findall(r'1[3-9]\d{9}', body)  # китайский мобильный номер
        emails = re.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', body)
        
        # Извлечение элементов дел (предложения, содержащие ключевые слова)
        todo_keywords = ['需要', '请', '准备', 'need', 'please', 'prepare', '确认', '完成', '提交']
        sentences = re.split(r'[。.!！]', body)
        todos = []
        for sentence in sentences:
            sentence = sentence.strip()
            if any(keyword in sentence for keyword in todo_keywords) and len(sentence) > 5:
                todos.append(sentence)
        
        # Ограничение количества задач
        todos = todos[:5]
        
        extracted_info = {
            'dates': list(set(dates)),  # удаление дубликатов
            'times': list(set(times)),
            'phones': list(set(phones)),
            'emails': list(set(emails)),
            'todos': todos
        }
        
        return json.dumps(extracted_info, ensure_ascii=False, indent=2)
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="body",
                type="string",
                description="Текст письма",
                required=True
            )
        ]

In [ ]:
class ReplyGenerationTool(Tool):
    """Инструмент генерации интеллектуальных ответов"""
    
    def __init__(self):
        super().__init__(
            name="reply_generation",
            description="Генерация подходящего черновика ответа на основе содержимого и классификации"
        )
        
        # Загрузка шаблонов ответов
        try:
            with open('templates/reply_templates.json', 'r', encoding='utf-8') as f:
                self.templates = json.load(f)
        except FileNotFoundError:
            # Шаблоны по умолчанию
            self.templates = {
                'work_meeting': {
                    'formal': {
                        'zh': 'Спасибо за письмо по теме {subject}. Отвечу подробно в течение 24 часов. При срочных вопросах свяжитесь со мной.\n\nС уважением',
                        'en': 'Thank you for your email regarding {subject}. I have received your information and will provide detailed feedback within 24 hours. Please feel free to contact me if there are any urgent matters.\n\nBest regards'
                    }
                },
                'customer_inquiry': {
                    'formal': {
                        'zh': 'Уважаемый клиент,\n\nблагодарим за интерес. По запросу {subject} ответим в течение 24 часов.\n\nС уважением',
                        'en': 'Dear Valued Customer,\n\nThank you for your interest in our products/services. Regarding your inquiry about {subject}, we will arrange for a professional to provide you with detailed answers within 24 hours.\n\nPlease feel free to contact us if you have any other questions.\n\nBest regards'
                    }
                },
                'general_acknowledgment': {
                    'formal': {
                        'zh': 'Здравствуйте,\n\nписьмо получено, отвечу в течение 24 часов.\n\nСпасибо!',
                        'en': 'Hello,\n\nI have received your email and will read it carefully and reply within 24 hours.\n\nThank you!'
                    }
                }
            }
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Генерация черновика ответа"""
        subject = parameters.get("subject", "")
        body = parameters.get("body", "")
        sender = parameters.get("sender", "")
        email_type = parameters.get("email_type", "other")
        
        if not subject and not body:
            return "Ошибка: тема и содержимое письма не могут быть пустыми одновременно"
        
        # Для спама ответ не генерируется
        if email_type == 'spam':
            return json.dumps({'message': 'Спам — ответ не генерируется'}, ensure_ascii=False)
        
        # Определить язык
        text_to_detect = f"{subject} {body}"
        try:
            detected_lang = detect(text_to_detect)
            is_chinese = detected_lang == 'zh-cn' or any('\u4e00' <= char <= '\u9fff' for char in text_to_detect)
        except:
            is_chinese = any('\u4e00' <= char <= '\u9fff' for char in text_to_detect)
        
        lang = 'zh' if is_chinese else 'en'
        
        # Выберите тип шаблона
        if email_type == 'work':
            template_key = 'work_meeting'
        elif email_type == 'customer':
            template_key = 'customer_inquiry'
        else:
            template_key = 'general_acknowledgment'
        
        # Получение шаблона
        template = self.templates.get(template_key, {}).get('formal', {}).get(lang, '')
        
        if not template:
            # Использование общего шаблона
            template = self.templates['general_acknowledgment']['formal'][lang]
        
        # Генерация текста ответа
        reply_content = template.format(
            subject=subject,
            timeframe='24часов' if lang == 'zh' else '24 hours'
        )
        
        reply_draft = {
            'to': sender,
            'subject': f"Re: {subject}",
            'content': reply_content,
            'language': lang,
            'template_type': template_key
        }
        
        return json.dumps(reply_draft, ensure_ascii=False, indent=2)
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="subject",
                type="string",
                description="Тема письма",
                required=False
            ),
            ToolParameter(
                name="body",
                type="string",
                description="Текст письма",
                required=False
            ),
            ToolParameter(
                name="sender",
                type="string",
                description="Адрес электронной почты отправителя",
                required=True
            ),
            ToolParameter(
                name="email_type",
                type="string",
                description="Тип классификации (work/customer/personal/spam/other)",
                required=False
            )
        ]

# ========================================
# 2. Создайте реестр инструментов и агент.# ========================================

In [ ]:
# Создание реестра инструментов
tool_registry = ToolRegistry()
tool_registry.register_tool(EmailClassificationTool())
tool_registry.register_tool(InfoExtractionTool())
tool_registry.register_tool(ReplyGenerationTool())

# Инициализация LLM
llm = HelloAgentsLLM()

# Определение слов системных подсказок
system_prompt = """Ты профессиональный помощник по обработке почты. Твои задачи:

1. Используй email_classification для анализа типа, приоритета и отправителя
2. Используй info_extraction для извлечения дат, времени, контактов и задач
3. Используй reply_generation для черновика ответа
4. На основе анализа предоставь подробный отчёт об обработке письма

Отчёт об обработке должен включать:
- Результаты классификации почты
- Извлечённую ключевую информацию
- Сгенерированный черновик ответа
- Рекомендации и напоминания

Выводи отчёт в структурированном формате на русском языке."""

# Создать агента
agent = SimpleAgent(
    name="Интеллектуальный помощник по электронной почте",
    llm=llm,
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

console.print("✅ Интеллектуальный помощник по электронной почтеИнициализация завершена!", style="green")

# ========================================
# 3. Запустите пример# ========================================

In [ ]:
# Примерные данные писем
sample_emails = [
    {
        'subject': 'Срочно: планирование встречи по статусу проекта',
        'sender': 'manager@company.com',
        'body': 'Коллеги, подготовьтесь к встрече по статусу проекта завтра в 14:00. Нужны итоги недели и план на следующую. Дедлайн: 2024-01-16 14:00. Подтвердите участие.'
    },
    {
        'subject': 'Запрос клиента: детали функций продукта',
        'sender': 'customer@client.com',
        'body': 'Здравствуйте, меня интересует ваш продукт. Можно ли назначить демонстрацию? Телефон: 13800138000. Жду ответа.'
    },
    {
        'subject': 'Urgent: Meeting Request',
        'sender': 'boss@company.com',
        'body': 'Hi team, we need to schedule an urgent meeting tomorrow at 3 PM to discuss the quarterly results. Please prepare your reports and confirm attendance by 5 PM today.'
    }
]

console.print(Panel.fit(
    f"📧 Подготовка к обработке {len(sample_emails)} примерных писем\n"
    "Включает рабочие письма, запросы клиентов и письма на английском",
    title="Начало обработки почты",
    style="blue"
))

In [ ]:
# Обработка каждого письма
results = []

for i, email in enumerate(sample_emails, 1):
    console.print(f"\n🔄 Обработка почты {i}/{len(sample_emails)}: {email['subject'][:30]}...", style="cyan")
    
    # Запрос на обработку сборки
    email_content = f"""
Обработайте следующее письмо:

Отправитель: {email['sender']}
Тема: {email['subject']}
Содержимое: {email['body']}

Выполните полный анализ и обработку письма.
"""
    
    # Выполнение обработки письма
    try:
        result = agent.run(email_content)
        results.append({
            'email': email,
            'result': result,
            'status': 'success'
        })
        console.print(f"✅ почта {i} Обработка завершена", style="green")
    except Exception as e:
        results.append({
            'email': email,
            'result': f"Обработка не удалась: {str(e)}",
            'status': 'error'
        })
        console.print(f"❌ почта {i} Обработка не удалась: {str(e)}", style="red")

console.print("\n🎉 Все письма обработаны!", style="bold green")

In [ ]:
# Отображение результатов обработки
console.print("\n" + "="*60)
console.print("📊 Сводка результатов обработки электронной почты", style="bold blue")
console.print("="*60)

success_count = sum(1 for r in results if r['status'] == 'success')
error_count = len(results) - success_count

# Создание статистических таблиц
stats_table = Table(title="Статистика процесса")
stats_table.add_column("проект", style="cyan")
stats_table.add_column("количество", style="white")

stats_table.add_row("Общее количество сообщений", str(len(results)))
stats_table.add_row("успешно обработано", str(success_count))
stats_table.add_row("Обработка не удалась", str(error_count))

console.print(stats_table)

# Показать подробные результаты
for i, result in enumerate(results, 1):
    if result['status'] == 'success':
        console.print(f"\n📧 почта {i} Результаты обработки:", style="bold yellow")
        console.print(f"Тема: {result['email']['subject']}")
        console.print(f"отправитель: {result['email']['sender']}")
        console.print("\nОтчёт об обработке:")
        console.print(result['result'])
        console.print("-" * 50)
    else:
        console.print(f"\n❌ почта {i} Обработка не удалась:", style="bold red")
        console.print(result['result'])

In [ ]:
# Сохранить отчет об обработке
import os
from datetime import datetime

# Убедитесь, что выходной каталог существует
os.makedirs('output/reports', exist_ok=True)

# Создать имя файла отчета
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_filename = f"output/reports/email_processing_report_{timestamp}.md"

# Создать отчет Markdown
report_content = f"""# Интеллектуальный помощник по электронной почтеОтчёт об обработке

**Время генерации**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
**Количество обработанных писем**: {len(results)}
**Успешно обработано**: {success_count}
**Обработка не удалась**: {error_count}

## Детали результата обработки

"""

for i, result in enumerate(results, 1):
    report_content += f"""### почта {i}

**Тема**: {result['email']['subject']}
**Отправитель**: {result['email']['sender']}
**Статус**: {'✅ успешно' if result['status'] == 'success' else '❌ неудача'}

**Результат обработки**:
```
{result['result']}
```

---

"""

# сохранить отчет
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write(report_content)

console.print(f"\n📄 Отчет об обработке сохранен в: {report_filename}", style="green")
console.print("\n💡 Предложения по следующим шагам:", style="blue")
console.print("1. Просмотр созданного черновика ответа")
console.print("2. Устанавливайте напоминания на основе извлеченной ключевой информации")
console.print("3. Настройте реальный почтовый ящик для фактической обработки электронной почты.")
console.print("4. Адаптируйте правила классификации и шаблоны ответов в соответствии с конкретными потребностями.")